# FASE 15 - EDA & Feature Engineering

**Track B: Machine Learning Predictions**

Goals:
- Load 5,432 historical predictions from FASE 14
- Exploratory Data Analysis (EDA)
- Feature Engineering Pipeline
- RandomForest Model Training
- SHAP Explainability

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import shap
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)
sns.set_style('whitegrid')

print('✅ Imports successful')

In [ ]:
# Load historical data from FASE 14
df = pd.read_csv('../data/fase14_predictions.csv')
print(f'📊 Loaded {len(df):,} records')
print(f'Columns: {list(df.columns)}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Data Info
print('📋 Data Info:')
print(df.info())
print('\n📊 Statistics:')
print(df.describe())
print('\n❓ Missing Values:')
print(df.isnull().sum())

In [ ]:
# Feature Engineering
def create_features(df):
    '''Create features for ML model'''
    features = df.copy()
    
    # Score averages
    features['avg_score'] = (features['web_score'] + 
                             features['facebook_score'] + 
                             features['google_score']) / 3
    
    # Score variance (consistency)
    score_cols = ['web_score', 'facebook_score', 'google_score']
    features['score_variance'] = features[score_cols].var(axis=1)
    
    # Business type dummies
    business_dummies = pd.get_dummies(features['business_type'], prefix='biz')
    features = pd.concat([features, business_dummies], axis=1)
    
    # Company size ordinal
    size_map = {'startup': 1, 'pyme': 2, 'empresa': 3}
    features['company_size_ord'] = features['company_size'].map(size_map)
    
    return features

df_features = create_features(df)
print(f'✨ Created {len(df_features.columns)} total features')
print(f'New features: {list(df_features.columns[-5:])}')

In [ ]:
# Train/Test Split
X = df_features[['web_score', 'facebook_score', 'google_score', 
                  'avg_score', 'score_variance', 'company_size_ord'] + 
                 [col for col in df_features.columns if col.startswith('biz_')]]
y = df['converted'].astype(int)  # Target: did they convert?

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'📈 Training set: {len(X_train):,} samples')
print(f'📉 Test set: {len(X_test):,} samples')
print(f'📊 Conversion rate: {y.mean():.1%}')

In [ ]:
# RandomForest Model Training
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print(f'🎯 Model Accuracy:')
print(f'  Training: {train_score:.1%}')
print(f'  Testing: {test_score:.1%}')
print(f'\n✅ Target achieved: >70% accuracy' if test_score > 0.70 else f'⚠️ Below 70% target')

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print('🎲 Feature Importance:')
print(feature_importance.head(10))

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
plt.title('Top 10 Most Important Features')
plt.xlabel('Importance')
plt.show()

In [ ]:
# SHAP Values for Explainability
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print('🔍 SHAP Values Generated')
print(f'  Test samples with explanations: {len(X_test)}')

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[1], X_test, show=False)  # Class 1 (conversion)
plt.title('SHAP Summary: Feature Impact on Conversion')
plt.show()

print('✅ SHAP explainability working')

In [ ]:
# Save Model & Scaler
import joblib

joblib.dump(model, '../models/predictor_model.pkl')

scaler = StandardScaler()
scaler.fit(X_train)
joblib.dump(scaler, '../models/feature_scaler.pkl')

print('💾 Model saved to ../models/predictor_model.pkl')
print('💾 Scaler saved to ../models/feature_scaler.pkl')